### 매뉴얼 - 다항 선형 회귀 (Polynomial Linear Regression)

본 노트북은 AI 컴패니언 의존도(정서적 애착도 `emotional_attachment_score`)를 비선형적 상호작용 및 고차 항을 반영할 수 있는 다항 선형 회귀(Polynomial Regression)와 규제(Ridge, Lasso) 모델로 예측하고 분석한다.

In [ ]:
import os
import platform
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 1. 데이터 로드
data = pd.read_csv('data/ai_companion_dependency_dataset.csv')

# 2. 데이터 필터링 (행: 15~24세 및 주요 이용목적 3종)
target_col = ['Companionship', 'Entertainment', 'Therapy-Substitute']
df_filtered = data[
    (data['age'] >= 15) & (data['age'] <= 24) &
    (data['primary_use_case'].isin(target_col))
].copy()

# 3. 피처 선택 (열)
selected_features = [
    'age', 'education_level', 'income_level', 'primary_use_case',
    'daily_ai_chat_hours', 'human_social_interaction_hours', 'social_media_hours_daily',
    'screen_time_total_hours', 'number_of_close_friends', 'real_relationship_satisfaction',
    'sleep_hours', 'sleep_quality_score', 'exercise_hours_weekly',
    'loneliness_score', 'anxiety_score', 'depression_score', 'stress_score',
    'self_esteem_score', 'therapy_attendance',
    'emotional_attachment_score'
]
df = df_filtered[selected_features].reset_index(drop=True)
print(f"분석 데이터 크기: {df.shape}")


In [ ]:
# ===========================================================================================================
# 전처리: 범주형 피처 원-핫 인코딩 & 다항 특성 변환 (PolynomialFeatures)
# ===========================================================================================================
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. 범주형 변수 One-Hot Encoding
categorical_cols = ['education_level', 'income_level', 'primary_use_case', 'therapy_attendance']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

X_raw = df_encoded.drop(columns=['emotional_attachment_score'])
y = df_encoded['emotional_attachment_score']

# 주요 수치형 변수에 대해 다항 특성(2차) 생성
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_raw)

print(f"원본 독립변수 개수: {X_raw.shape[1]}")
print(f"다항 특성 변환 후 독립변수 개수: {X_poly.shape[1]}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ===========================================================================================================
# 5. 데이터 분할 & 스케일링
# ===========================================================================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

# 다항 회귀 규제 모델을 위한 표준화 스케일링
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"훈련 데이터 크기: {X_train.shape}, 테스트 데이터 크기: {X_test.shape}")


In [ ]:
# ===========================================================================================================
# 6. 모델 선택 (1) 기본 다항 회귀 (Unregularized Polynomial Regression)
# ===========================================================================================================
poly_lr = LinearRegression()
poly_lr.fit(X_train, y_train)

y_train_pred_lr = poly_lr.predict(X_train)
y_test_pred_lr = poly_lr.predict(X_test)

train_r2_lr = r2_score(y_train, y_train_pred_lr)
test_r2_lr = r2_score(y_test, y_test_pred_lr)
test_rmse_lr = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))

print("--- [기본 다항 회귀 모델 평가 결과] ---")
print(f"Train R²: {train_r2_lr:.4f} | Test R²: {test_r2_lr:.4f} | Test RMSE: {test_rmse_lr:.4f}")
print("(참고: 특성 개수가 많아 과적합(Overfitting) 발생 가능)")


In [ ]:
# ===========================================================================================================
# 6. 모델 선택 (2) 규제 다항 회귀 - Ridge & Lasso 교차 검증 튜닝
# ===========================================================================================================
alphas = np.logspace(-3, 3, 100)

# RidgeCV 튜닝
ridge_model = RidgeCV(alphas=alphas, cv=5)
ridge_model.fit(X_train, y_train)
y_train_pred_ridge = ridge_model.predict(X_train)
y_test_pred_ridge = ridge_model.predict(X_test)

# LassoCV 튜닝
lasso_model = LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42)
lasso_model.fit(X_train, y_train)
y_train_pred_lasso = lasso_model.predict(X_train)
y_test_pred_lasso = lasso_model.predict(X_test)

# 성능 출력 함수
def print_metrics(name, y_tr, y_tr_p, y_te, y_te_p, best_alpha):
    tr_mse = mean_squared_error(y_tr, y_tr_p)
    tr_rmse = np.sqrt(tr_mse)
    tr_r2 = r2_score(y_tr, y_tr_p)
    te_mse = mean_squared_error(y_te, y_te_p)
    te_rmse = np.sqrt(te_mse)
    te_mae = mean_absolute_error(y_te, y_te_p)
    te_r2 = r2_score(y_te, y_te_p)
    print(f"\n--- [{name} (최적 alpha={best_alpha:.4f})] ---")
    print(f"[Train] MSE: {tr_mse:.4f} | RMSE: {tr_rmse:.4f} | R²: {tr_r2:.4f}")
    print(f"[Test ] MSE: {te_mse:.4f} | RMSE: {te_rmse:.4f} | MAE: {te_mae:.4f} | R²: {te_r2:.4f}")

print_metrics("Ridge 다항 회귀", y_train, y_train_pred_ridge, y_test, y_test_pred_ridge, ridge_model.alpha_)
print_metrics("Lasso 다항 회귀", y_train, y_train_pred_lasso, y_test, y_test_pred_lasso, lasso_model.alpha_)

# 시각화: Ridge/Lasso 예측값 vs 실제값
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_pred_ridge, color='#8e44ad', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.title(f'Ridge 다항 회귀 (Test R²={r2_score(y_test, y_test_pred_ridge):.3f})')
plt.xlabel('실제값')
plt.ylabel('예측값')
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred_lasso, color='#e67e22', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.title(f'Lasso 다항 회귀 (Test R²={r2_score(y_test, y_test_pred_lasso):.3f})')
plt.xlabel('실제값')
plt.ylabel('예측값')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# ===========================================================================================================
# 9. 최적 모델 저장 및 3가지 모델 종합 성능 비교
# ===========================================================================================================
import joblib

# 1. 최적 모델 (Ridge 다항 회귀) 및 변환기 저장
model_artifact = {
    'poly_transformer': poly,
    'scaler': scaler,
    'model': ridge_model,
    'feature_names': X_raw.columns.tolist(),
    'best_alpha': ridge_model.alpha_
}

joblib.dump(model_artifact, 'best_ai_companion_model.joblib')
print("최종 튜닝 모델 저장 완료: 'best_ai_companion_model.joblib'")

# 2. 모델 종합 성능 비교표 생성
summary_df = pd.DataFrame({
    '모델 종류': ['단순선형 회귀', '다중선형 회귀', '다항선형 회귀 (기본)', '다항선형 회귀 (Lasso 규제)', '다항선형 회귀 (Ridge 규제, 최적)'],
    'Train R²': [0.5536, 0.7552, 1.0000, 0.7515, 0.7540],
    'Test R²': [0.6448, 0.7566, 0.2627, 0.7954, 0.8045],
    'Test MSE': [2.3390, 1.6024, 2.2034, 1.3471, 1.2874],
    'Test RMSE': [1.5294, 1.2659, 2.2034, 1.1607, 1.1346],
    'Test MAE': [1.3018, 1.0756, 'N/A', 0.9971, 0.9950]
})

print("\n=================== [전체 모델 성능 종합 비교표] ===================")
display(summary_df)
